# AutismInsight NLP — Data Preparation & TF-IDF Baseline

**Course project:** Multi-task classification of autism research and scientific evidence using NLP.

**Member 1 responsibility:** Collect and prepare PubMed abstracts, coordinate topic-label validation, and develop a TF-IDF + Logistic Regression baseline.

**Current status:** Data collection and proposed annotation assignments completed. Human labeling and baseline model training remain pending.


## 2. Research Question & Scope

**Question:** Can a TF-IDF + Logistic Regression baseline classify autism-related research abstracts into Genetics, Neuroscience, Immunology, Metabolism, or Interventions?

Topic labels must be validated by human reviewers before supervised training. Evidence/uncertainty classification is a separate team task requiring separate annotations.

## 3. Environment & Imports

Run the import cell, then the portable configuration cell. Set a valid NCBI contact email before any fresh PubMed retrieval.

In [1]:
# ============================================================
#  Team 3
# AutismInsight NLP
# Dataset Preparation & Baseline Classification
# ============================================================

# Data manipulation
import pandas as pd
import numpy as np

# PubMed data collection
from Bio import Entrez

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Visualization
import matplotlib.pyplot as plt

# Utilities
import time
from pathlib import Path

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [2]:
# ============================================================
# Project Configuration — portable across operating systems
# ============================================================

project_dir = Path.cwd()
if project_dir.name.lower() == "notebooks":
    project_dir = project_dir.parent

if not (project_dir / "Data").is_dir():
    raise FileNotFoundError(
        "Project Data folder not found. Launch Jupyter from "
        "AutismInsight_NLP/ or AutismInsight_NLP/Notebooks/."
    )

data_dir = project_dir / "Data"
raw_dir = data_dir / "Raw"
processed_dir = data_dir / "Processed"
models_dir = project_dir / "Models"
results_dir = project_dir / "Results"
documentation_dir = project_dir / "Documentation"

for directory in [raw_dir, processed_dir, models_dir, results_dir, documentation_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory:", project_dir)


Project directory: D:\Natural Lang\AutismInsight_NLP


## 4. Data Sources & Provenance

Source: NCBI PubMed. The pilot searched for up to 10 articles per category; the expanded collection searched for up to 200 per category. Results depend on the search date and PubMed ranking. Existing CSV files should be reused rather than downloading again unnecessarily.

In [3]:


# Identify ourselves to the NCBI PubMed service
Entrez.email = "your_email@example.com"

# Define our five autism research categories
categories = {
    "Genetics": "autism AND genetics",
    "Neuroscience": "autism AND brain development",
    "Immunology": "autism AND inflammation",
    "Metabolism": "autism AND mitochondrial function",
    "Interventions": "autism AND behavioral intervention"
}

# Search PubMed for 10 articles per category
records = []

for category, query in categories.items():

    handle = Entrez.esearch(
        db="pubmed",
        term=f"({query}) AND hasabstract[text]",
        retmax=10,
        sort="relevance"
    )

    results = Entrez.read(handle)
    handle.close()

    for pmid in results["IdList"]:
        records.append({
            "PMID": str(pmid),
            "Category": category
        })

df = pd.DataFrame(records)

print("Total candidate records:", len(df))
display(df.head(10))

Total candidate records: 50


,PMID,Category
0,23226953,Genetics
1,28260791,Genetics
2,32940822,Genetics
3,36368308,Genetics
4,31097668,Genetics
5,30776274,Genetics
6,38958063,Genetics
7,22851255,Genetics
8,38696597,Genetics
9,16791389,Genetics


In [4]:

# Retrieve article information from PubMed
pmids = df["PMID"].tolist()

handle = Entrez.efetch(
    db="pubmed",
    id=",".join(pmids),
    rettype="medline",
    retmode="xml"
)

articles = Entrez.read(handle)
handle.close()

# Store article details
article_data = []

for article in articles["PubmedArticle"]:

    citation = article["MedlineCitation"]
    pmid = str(citation["PMID"])
    article_info = citation["Article"]

    title = str(article_info["ArticleTitle"])

    # Extract abstract
    abstract = ""

    if "Abstract" in article_info:
        abstract = " ".join(
            str(text)
            for text in article_info["Abstract"]["AbstractText"]
        )

    article_data.append({
        "PMID": pmid,
        "Title": title,
        "Abstract": abstract
    })

# Create DataFrame
abstract_df = pd.DataFrame(article_data)

# Add provisional research categories
abstract_df = abstract_df.merge(
    df,
    on="PMID",
    how="left"
)

print("Total records:", len(abstract_df))
print("Records with abstracts:",
      (abstract_df["Abstract"].str.len() > 0).sum())

display(abstract_df.head())

Total records: 50
Records with abstracts: 50


,PMID,Title,Abstract,Category
0,23226953,"Autism risk factors: genes, environment, and g...",The aim of this review is to summarize the key...,Genetics
1,28260791,Autism genetics: opportunities and challenges ...,Genetic studies have revealed the involvement ...,Genetics
2,32940822,Genetic Advances in Autism.,"In the last 40 years, there has been a huge in...",Genetics
3,31097668,Single-cell genomics identifies cell type-spec...,Despite the clinical and genetic heterogeneity...,Genetics
4,30776274,[Autism. Genetic and biological aspects].,Autism is a neurodevelopmental disorder charac...,Genetics


In [5]:
# ============================================================
# Step 4: Retrieve PubMed Article Titles and Abstracts
# ============================================================

# Get PubMed IDs from our search results
pmids = df["PMID"].unique().tolist()

# Retrieve article information
with Entrez.efetch(
    db="pubmed",
    id=",".join(pmids),
    retmode="xml"
) as handle:
    articles = Entrez.read(handle)

# Extract article details
article_data = []

for article in articles["PubmedArticle"]:

    citation = article["MedlineCitation"]
    pmid = str(citation["PMID"])
    article_info = citation["Article"]

    title = str(article_info["ArticleTitle"])

    # Extract abstract text
    abstract = ""

    if "Abstract" in article_info:
        abstract = " ".join(
            str(text)
            for text in article_info["Abstract"]["AbstractText"]
        )

    article_data.append({
        "PMID": pmid,
        "Title": title,
        "Abstract": abstract
    })

# Create DataFrame
abstract_df = pd.DataFrame(article_data)

# Merge with provisional research categories
abstract_df = abstract_df.merge(
    df,
    on="PMID",
    how="left"
)

# Display results
print("Total records:", len(abstract_df))
print(
    "Records with abstracts:",
    (abstract_df["Abstract"].str.len() > 0).sum()
)

display(abstract_df.head())

Total records: 50
Records with abstracts: 50


,PMID,Title,Abstract,Category
0,23226953,"Autism risk factors: genes, environment, and g...",The aim of this review is to summarize the key...,Genetics
1,28260791,Autism genetics: opportunities and challenges ...,Genetic studies have revealed the involvement ...,Genetics
2,32940822,Genetic Advances in Autism.,"In the last 40 years, there has been a huge in...",Genetics
3,36368308,Genomic architecture of autism from comprehens...,Fully understanding autism spectrum disorder (...,Genetics
4,36368308,Genomic architecture of autism from comprehens...,Fully understanding autism spectrum disorder (...,Metabolism


### Expanded PubMed collection

The expanded search produced 1,000 category-search records and 956 unique PMIDs in the original run.

In [6]:
# ============================================================
# Research Category Configuration
# ============================================================

research_queries = {
    "Genetics": "autism AND genetics",
    "Neuroscience": "autism AND brain development",
    "Immunology": "autism AND inflammation",
    "Metabolism": "autism AND mitochondrial function",
    "Interventions": "autism AND behavioral intervention"
}

print("Research categories configured successfully!")

Research categories configured successfully!


## 10. Expand PubMed Research Dataset

In [7]:
# ============================================================
# Step 10: Expand PubMed Research Dataset
# ============================================================

expanded_records = []

for category, query in research_queries.items():

    with Entrez.esearch(
        db="pubmed",
        term=f"({query}) AND hasabstract[text]",
        retmax=200,
        sort="relevance"
    ) as handle:
        results = Entrez.read(handle)

    for pmid in results["IdList"]:
        expanded_records.append({
            "PMID": str(pmid),
            "Category": category
        })

    print(f"{category}: {len(results['IdList'])} articles")

    time.sleep(0.4)

expanded_df = pd.DataFrame(expanded_records)

print("\nTotal candidate records:", len(expanded_df))
print("Unique articles:", expanded_df["PMID"].nunique())
print("Overlapping records:", expanded_df["PMID"].duplicated().sum())

Genetics: 200 articles
Neuroscience: 200 articles
Immunology: 200 articles
Metabolism: 200 articles
Interventions: 200 articles

Total candidate records: 1000
Unique articles: 957
Overlapping records: 43


## 11. Retrieve Expanded PubMed Abstracts

In [8]:
# ============================================================
# Step 11: Retrieve Expanded PubMed Abstracts
# ============================================================

# Get unique PubMed IDs
unique_pmids = expanded_df["PMID"].unique().tolist()

# Download abstracts in batches
batch_size = 100
all_articles = []

for start in range(0, len(unique_pmids), batch_size):

    batch = unique_pmids[start:start + batch_size]

    with Entrez.efetch(
        db="pubmed",
        id=",".join(batch),
        retmode="xml"
    ) as handle:
        results = Entrez.read(handle)

    for article in results["PubmedArticle"]:

        citation = article["MedlineCitation"]
        pmid = str(citation["PMID"])
        article_info = citation["Article"]

        title = str(article_info["ArticleTitle"])

        abstract = ""

        if "Abstract" in article_info:
            abstract = " ".join(
                str(text)
                for text in article_info["Abstract"]["AbstractText"]
            )

        all_articles.append({
            "PMID": pmid,
            "Title": title,
            "Abstract": abstract
        })

    print(f"Processed {min(start + batch_size, len(unique_pmids))} / {len(unique_pmids)} articles")

    time.sleep(0.4)

# Create expanded dataset
full_df = pd.DataFrame(all_articles)

# Preserve all candidate category assignments
full_df = full_df.merge(
    expanded_df,
    on="PMID",
    how="left"
)

print("\nDATA COLLECTION SUMMARY")
print("=" * 45)
print("Total records:", len(full_df))
print("Unique articles:", full_df["PMID"].nunique())
print("Articles with abstracts:",
      full_df.loc[
          full_df["Abstract"].str.strip() != "", "PMID"
      ].nunique())

Processed 100 / 957 articles
Processed 200 / 957 articles
Processed 300 / 957 articles
Processed 400 / 957 articles
Processed 500 / 957 articles
Processed 600 / 957 articles
Processed 700 / 957 articles
Processed 800 / 957 articles
Processed 900 / 957 articles
Processed 957 / 957 articles

DATA COLLECTION SUMMARY
Total records: 1000
Unique articles: 957
Articles with abstracts: 957


## 12. Save Expanded PubMed Dataset

In [9]:
# ============================================================
# Step 12: Save Expanded PubMed Dataset
# ============================================================

# Save the original expanded dataset
output_file = raw_dir / "autisminsight_raw_1000.csv"

full_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("Dataset saved successfully!")
print("File location:", output_file.resolve())
print("Total records:", len(full_df))
print("Unique articles:", full_df["PMID"].nunique())

Dataset saved successfully!
File location: D:\Natural Lang\AutismInsight_NLP\Data\Raw\autisminsight_raw_1000.csv
Total records: 1000
Unique articles: 957


## 5. Data Quality & Annotation

Search categories are provisional. Human review determines the final primary topic label. See `Documentation/Annotation_Guidelines.md`.

## 5. Data Cleaning and Quality Assessment

In [10]:
# ============================================================
# Step 5: Data Cleaning and Quality Assessment
# ============================================================

print("DATA QUALITY REPORT")
print("=" * 45)

# Total records
print("Total records:", len(abstract_df))

# Check duplicate PubMed IDs
duplicate_count = abstract_df["PMID"].duplicated().sum()
print("Duplicate PMIDs:", duplicate_count)

# Check missing abstracts
missing_abstracts = (
    abstract_df["Abstract"].isna() |
    (abstract_df["Abstract"].str.strip() == "")
).sum()

print("Missing abstracts:", missing_abstracts)

# Check missing titles
missing_titles = (
    abstract_df["Title"].isna() |
    (abstract_df["Title"].str.strip() == "")
).sum()

print("Missing titles:", missing_titles)

# Check category distribution
print("\nCATEGORY DISTRIBUTION")
print("=" * 45)

print(abstract_df["Category"].value_counts())

# Identify articles assigned to multiple categories
multi_category = (
    abstract_df.groupby("PMID")["Category"]
    .nunique()
)

print("\nArticles appearing in multiple categories:",
      (multi_category > 1).sum())

DATA QUALITY REPORT
Total records: 50
Duplicate PMIDs: 4
Missing abstracts: 0
Missing titles: 0

CATEGORY DISTRIBUTION
Category
Genetics         10
Metabolism       10
Interventions    10
Neuroscience     10
Immunology       10
Name: count, dtype: int64

Articles appearing in multiple categories: 4


## 6. Identify Overlapping Research Categories

In [11]:
# ============================================================
# Step 6: Identify Articles in Multiple Categories
# ============================================================

overlapping_articles = (
    abstract_df.groupby(["PMID", "Title"])["Category"]
    .agg(lambda categories: ", ".join(sorted(set(categories))))
    .reset_index()
)

overlapping_articles = overlapping_articles[
    overlapping_articles["Category"].str.contains(",")
]

print("Articles with overlapping categories:", len(overlapping_articles))

display(overlapping_articles)

Articles with overlapping categories: 4


,PMID,Title,Category
3,18606031,"Early behavioral intervention, brain plasticit...","Interventions, Neuroscience"
8,22851255,ADHD and autism: differential diagnosis or ove...,"Genetics, Interventions"
27,36368308,Genomic architecture of autism from comprehens...,"Genetics, Metabolism"
32,38696597,Neuroanatomy of autism: what is the role of th...,"Genetics, Neuroscience"


## 7. Review Ambiguous Research Articles

In [12]:
# ============================================================
# Step 7: Review Articles with Overlapping Categories
# ============================================================

overlap_pmids = overlapping_articles["PMID"].tolist()

for pmid in overlap_pmids:

    article = abstract_df[
        abstract_df["PMID"] == pmid
    ].iloc[0]

    categories = abstract_df.loc[
        abstract_df["PMID"] == pmid, "Category"
    ].unique()

    print("=" * 70)
    print("PMID:", pmid)
    print("TITLE:", article["Title"])
    print("CATEGORIES:", ", ".join(categories))
    print("\nABSTRACT:")
    print(article["Abstract"])
    print()
    

PMID: 18606031
TITLE: Early behavioral intervention, brain plasticity, and the prevention of autism spectrum disorder.
CATEGORIES: Neuroscience, Interventions

ABSTRACT:
Advances in the fields of cognitive and affective developmental neuroscience, developmental psychopathology, neurobiology, genetics, and applied behavior analysis have contributed to a more optimistic outcome for individuals with autism spectrum disorder (ASD). These advances have led to new methods for early detection and more effective treatments. For the first time, prevention of ASD is plausible. Prevention will entail detecting infants at risk before the full syndrome is present and implementing treatments designed to alter the course of early behavioral and brain development. This article describes a developmental model of risk, risk processes, symptom emergence, and adaptation in ASD that offers a framework for understanding early brain plasticity in ASD and its role in prevention of the disorder.

PMID: 2285125

## 8. Create Cleaned and Reviewed Pilot Dataset

In [13]:
# ============================================================
# Step 8: Create Cleaned and Reviewed Pilot Dataset
# ============================================================

# Preserve original dataset
clean_df = abstract_df.copy()

# Remove duplicate articles
clean_df = clean_df.drop_duplicates(subset="PMID").copy()

# Apply manually reviewed primary categories
reviewed_labels = {
    "18606031": "Interventions",
    "36368308": "Genetics",
    "38696597": "Neuroscience"
}

for pmid, category in reviewed_labels.items():
    clean_df.loc[
        clean_df["PMID"] == pmid, "Category"
    ] = category

# Exclude article outside our research scope
clean_df = clean_df[
    clean_df["PMID"] != "22851255"
].copy()

# Remove records without abstracts
clean_df = clean_df[
    clean_df["Abstract"].notna() &
    (clean_df["Abstract"].str.strip() != "")
]

# Reset index
clean_df = clean_df.reset_index(drop=True)

# Display cleaned dataset summary
print("CLEANED DATASET REPORT")
print("=" * 45)

print("Total articles:", len(clean_df))
print("Duplicate PMIDs:", clean_df["PMID"].duplicated().sum())
print("Missing abstracts:", clean_df["Abstract"].isna().sum())

print("\nCATEGORY DISTRIBUTION")
print(clean_df["Category"].value_counts())

display(clean_df.head())

CLEANED DATASET REPORT
Total articles: 45
Duplicate PMIDs: 0
Missing abstracts: 0

CATEGORY DISTRIBUTION
Category
Immunology       10
Neuroscience      9
Interventions     9
Metabolism        9
Genetics          8
Name: count, dtype: int64


,PMID,Title,Abstract,Category
0,23226953,"Autism risk factors: genes, environment, and g...",The aim of this review is to summarize the key...,Genetics
1,28260791,Autism genetics: opportunities and challenges ...,Genetic studies have revealed the involvement ...,Genetics
2,32940822,Genetic Advances in Autism.,"In the last 40 years, there has been a huge in...",Genetics
3,36368308,Genomic architecture of autism from comprehens...,Fully understanding autism spectrum disorder (...,Genetics
4,31097668,Single-cell genomics identifies cell type-spec...,Despite the clinical and genetic heterogeneity...,Genetics


## 9. Save Cleaned Pilot Dataset

In [14]:
# ============================================================
# Step 9: Save Cleaned Pilot Dataset
# ============================================================

output_file = processed_dir / "autisminsight_pilot_45.csv"

clean_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("Pilot dataset saved successfully!")
print("File location:", output_file.resolve())
print("Total articles:", len(clean_df))

Pilot dataset saved successfully!
File location: D:\Natural Lang\AutismInsight_NLP\Data\Processed\autisminsight_pilot_45.csv
Total articles: 45


## 13. Identify Overlapping Research Categories


In [15]:
# ============================================================
# Step 13: Identify Overlapping Research Categories
# ============================================================

# Identify articles assigned to multiple categories
category_overlap = (
    full_df.groupby("PMID")
    .agg(
        Title=("Title", "first"),
        Categories=("Category", lambda x: ", ".join(sorted(set(x)))),
        Category_Count=("Category", "nunique")
    )
    .reset_index()
)

# Keep articles assigned to multiple categories
overlapping_df = category_overlap[
    category_overlap["Category_Count"] > 1
].copy()

# Display summary
print("OVERLAPPING CATEGORY REPORT")
print("=" * 45)
print("Unique articles:", full_df["PMID"].nunique())
print("Articles with multiple categories:", len(overlapping_df))

display(overlapping_df.head(10))

OVERLAPPING CATEGORY REPORT
Unique articles: 957
Articles with multiple categories: 41


,PMID,Title,Categories,Category_Count
5,10532625,Neurodevelopmental processes and psychological...,"Genetics, Neuroscience",2
57,14574828,Genetic and neurodevelopmental influences in a...,"Genetics, Neuroscience",2
93,16264412,Mercury and autism: accelerating evidence?,"Immunology, Neuroscience",2
95,16280193,Macrocephaly and the control of brain growth i...,"Genetics, Neuroscience",2
124,17084999,Childhood autism and associated comorbidities.,"Genetics, Neuroscience",2
125,17236739,"A review of gene linkage, association and expr...","Genetics, Neuroscience",2
126,17275283,Autism spectrum disorders: developmental disco...,"Genetics, Neuroscience",2
140,17920208,Autism and vitamin D.,"Genetics, Immunology, Neuroscience",3
151,18317271,What is new in autism?,"Genetics, Neuroscience",2
160,18606031,"Early behavioral intervention, brain plasticit...","Interventions, Neuroscience",2


## Prepare Dataset for Research Category Validation

Create one row per PMID. Candidate categories are search-derived suggestions, **not validated ground-truth labels**.

In [16]:
# ============================================================
# Step 14: Prepare Dataset for Category Validation
# ============================================================

# Create one row per unique research article
review_df = (
    full_df.groupby("PMID", as_index=False)
    .agg(
        Title=("Title", "first"),
        Abstract=("Abstract", "first"),
        Candidate_Categories=(
            "Category",
            lambda x: ", ".join(sorted(set(x)))
        )
    )
)

# Add columns for manual annotation
review_df["Primary_Category"] = ""
review_df["Review_Status"] = "Pending"
review_df["Reviewer_Notes"] = ""

# Save review dataset
review_file = data_dir / "autisminsight_label_review.csv"

review_df.to_csv(
    review_file,
    index=False,
    encoding="utf-8"
)

print("LABEL REVIEW DATASET")
print("=" * 45)
print("Total unique articles:", len(review_df))
print("Duplicate PMIDs:", review_df["PMID"].duplicated().sum())
print("Review file:", review_file.resolve())

display(review_df.head())

LABEL REVIEW DATASET
Total unique articles: 957
Duplicate PMIDs: 0
Review file: D:\Natural Lang\AutismInsight_NLP\Data\autisminsight_label_review.csv


,PMID,Title,Abstract,Candidate_Categories,Primary_Category,Review_Status,Reviewer_Notes
0,10102725,Genetics and child psychiatry: II Empirical re...,Key substantive findings from quantitative and...,Genetics,,Pending,
1,10190270,Genetic susceptibility to neurodevelopmental d...,A large body of evidence suggests that genetic...,Genetics,,Pending,
2,10212552,Etiology of infantile autism: a review of rece...,"The etiology of autism is complex, and in most...",Neuroscience,,Pending,
3,10212560,Heterogeneity and the genetics of autism.,The objective of this review is to summarize r...,Genetics,,Pending,
4,10226747,Autism and the pervasive developmental disorders.,Our understanding of childhood autism and the ...,Genetics,,Pending,


## 15. Research Category Annotation Guidelines

### Objective
Assign each autism research abstract one primary research category
based on its central scientific focus.

### Research Categories

| Category | Definition |

| Genetics | Genetic variants, genes, heredity, genomic sequencing, and genetic risk factors |
| Neuroscience | Brain structure, neural development, neuroimaging, and neurological mechanisms |
| Immunology | Immune responses, inflammation, cytokines, and neuroimmune mechanisms |
| Metabolism | Mitochondrial function, metabolic pathways, energy production, and metabolic abnormalities |
| Interventions | Behavioral therapies, developmental interventions, and treatment outcomes |

### Annotation Rules

1. Read the title and complete abstract.
2. Identify the article's primary research question.
3. Assign the category that best represents its central focus.
4. Do not assign a category solely because a keyword appears.
5. If multiple categories apply, select the dominant research focus.
6. If no category adequately represents the article, mark it as Excluded.
7. Record ambiguous decisions in Reviewer_Notes.
8. Use Review_Status = Reviewed only after examining the abstract.

### Quality Assurance

- Review all overlapping category assignments.
- Validate single-category search results rather than assuming they are correct.
- Have a second reviewer independently annotate a representative sample.
- Resolve disagreements before finalizing labels.
- Keep a record of excluded articles and labeling decisions.

### Important Limitation

Candidate categories originate from PubMed search queries and
are not independently verified ground-truth labels.

## 16. Prepare Team Annotation Assignments

In [17]:
# ============================================================
# Step 16: Prepare Team Annotation Assignments
# ============================================================

# Shuffle articles reproducibly
shuffled_df = review_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Divide articles among three reviewers
assignments = np.array_split(
    shuffled_df.index.to_numpy(), 3
)

# Save each member's assigned articles
for member, indices in enumerate(assignments, start=1):

    member_df = shuffled_df.loc[indices].copy()

    member_df["Assigned_Reviewer"] = f"Member {member}"

    file_path = data_dir / f"annotation_member_{member}.csv"

    member_df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Member {member}: {len(member_df)} articles")

print("\nAnnotation files created successfully!")

Member 1: 319 articles
Member 2: 319 articles
Member 3: 319 articles

Annotation files created successfully!


**Annotation safeguards:** The three reviewer CSVs are proposed assignments, not completed annotations. Preserve edited reviewer files before rerunning the assignment-generation cell, because it overwrites those CSVs. Double-label a subset independently, calculate pairwise agreement, resolve disagreements, and save adjudicated final labels.

## 17. Organize Shared Project Files

In [18]:
# ============================================================
# Step 17: Verify Shared Project Folders
# ============================================================

project_folders = [
    data_dir, raw_dir, processed_dir,
    project_dir / "Notebooks",
    models_dir, results_dir, documentation_dir
]

for folder in project_folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(folder.relative_to(project_dir), "— ready")


Data — ready
Data\Raw — ready
Data\Processed — ready
Notebooks — ready
Models — ready
Results — ready
Documentation — ready


## 6. Dataset Splits — pending validated labels

After adjudication, create a single labeled dataset and a reproducible stratified train/validation/test split (for example, 70/15/15). Fit TF-IDF **only on training data** to prevent leakage. Save split assignments so the transformer uses the same examples.

In [19]:
# TODO: Load adjudicated topic labels, check class counts,
# and create reproducible train/validation/test splits.
# Do not train using provisional PubMed search categories.

## 7. Method & Implementation — TF-IDF + Logistic Regression

Train a supervised baseline only after final labels and shared splits exist.

In [20]:
# TODO: Fit TfidfVectorizer on training abstracts only.
# TODO: Train LogisticRegression on validated topic labels.
# TODO: Save fitted vectorizer and classifier to Models/.

## 8. Evaluation

Report accuracy, macro-averaged precision/recall/F1, per-class scores, and a confusion matrix on held-out test data.

In [21]:
# TODO: Evaluate the frozen model on the held-out test set.
# TODO: Save classification report and confusion matrix to Results/.

## 9. Results & Error Analysis

**Pending model training.** Document the baseline metrics, common confusions between categories, and representative errors.

## 10. Ethics & Limitations

PubMed search results are not representative of all autism research. The five topics can overlap; forced single-label classification loses nuance. Manual annotations may be subjective. These models summarize research topics and must not be used for clinical diagnosis or treatment decisions.

## 11. Reproducibility & Team Handoff

- Run from the project root or `Notebooks/` folder.
- Keep data under `Data/`, models under `Models/`, and metrics under `Results/`.
- Use the validated labels and **identical saved splits** for baseline and transformer comparisons.
- Member 2: transformer topic classification.
- Member 3: evidence/uncertainty classification with separately validated labels.
- Document package versions, random seeds, data provenance, and evaluation settings.
- Avoid rerunning collection/annotation-export cells when working with edited files.